# 🚦 B4DCrash — Road Safety Near-Miss Detection Pipeline

**End-to-end execution on Kaggle GPU (T4 / P100)**

Pipeline stages:
```
Video → YOLO Detection → ByteTrack → Trajectory Analysis
     → TTC/PET Safety Metrics → Conflict Events
     → Hotspot Aggregation → Agent37 Safety Intelligence
```

**⚠️ Before running:** Enable **GPU T4 x2** or **P100** in Notebook Settings (right panel → Accelerator).

---

## Step 0 — Environment Check

In [ ]:
import os, sys, platform, subprocess
from pathlib import Path

print('Python  :', sys.version)
print('Platform:', platform.system(), platform.machine())

try:
    import torch
    cuda_ok = torch.cuda.is_available()
    device   = torch.cuda.get_device_name(0) if cuda_ok else 'CPU'
    print('PyTorch :', torch.__version__)
    if cuda_ok:
        print('CUDA    : ✅', device, '(CUDA', torch.version.cuda, ')')
    else:
        print('CUDA    : ❌ No GPU — enable GPU Accelerator in Notebook settings before proceeding.')
        raise SystemExit('Enable GPU and re-run.')
except ImportError:
    print('PyTorch not installed yet — will install in Step 1.')


## Step 1 — Install Dependencies

> `lap` must be installed **before** `ultralytics` to avoid the ByteTrack restart-required bug.


In [ ]:
# lap first — ByteTrack linear assignment
!pip install -q 'lap>=0.5.12'

# Core ML / vision stack
!pip install -q \
    'ultralytics>=8.2' \
    'opencv-python-headless>=4.9' \
    'scipy>=1.12' \
    'pandas>=2.2' \
    'PyYAML>=6.0' \
    'tqdm>=4.66' \
    'scikit-learn>=1.4' \
    'matplotlib>=3.8' \
    'seaborn>=0.13'

print('✅ Dependencies installed.')


## Step 2 — Clone B4DCrash Repository

In [ ]:
REPO_URL  = 'https://github.com/ShraddhaBora/B4D-Crash.git'
REPO_ROOT = Path('/kaggle/working/B4D-Crash')

if not REPO_ROOT.exists():
    print('Cloning', REPO_URL, '...')
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
    print('✅ Cloned to', REPO_ROOT)
else:
    print('✅ Repo already present at', REPO_ROOT)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print('CWD:', os.getcwd())
print('src/pipeline.py exists:', (REPO_ROOT / 'src' / 'pipeline.py').exists())


## Step 3 — Acquire a Sample Traffic Video

Choose **one** option:

| Option | Method | Notes |
|--------|--------|-------|
| **A** | Download from public URL | Requires Internet ON |
| **B** | Kaggle-attached dataset | Add via right panel > Add Data |
| **C** | Synthetic video (CPU, no download) | Validates pipeline structure offline |


In [ ]:
# === OPTION A: Download from public URL =====================================
# Replace VIDEO_URL with any publicly accessible .mp4 link.
# Example traffic datasets on Kaggle:
#   https://www.kaggle.com/datasets/aryashah2k/highway-traffic-videos-dataset
#   https://www.kaggle.com/datasets/robjfalkner/video-surveillance

VIDEO_DIR  = Path('data/samples')
VIDEO_DIR.mkdir(parents=True, exist_ok=True)
VIDEO_PATH = str(VIDEO_DIR / 'sample_traffic.mp4')

VIDEO_URL = ''  # <-- paste your URL here

if VIDEO_URL and not Path(VIDEO_PATH).exists():
    print('Downloading video from:', VIDEO_URL)
    subprocess.run(['wget', '-q', '-O', VIDEO_PATH, VIDEO_URL], check=True)
    sz = Path(VIDEO_PATH).stat().st_size / 1024**2
    print(f'✅ Saved: {VIDEO_PATH} ({sz:.1f} MB)')
elif Path(VIDEO_PATH).exists():
    sz = Path(VIDEO_PATH).stat().st_size / 1024**2
    print(f'✅ Video already present: {VIDEO_PATH} ({sz:.1f} MB)')
else:
    print('⚠️  VIDEO_URL is empty. Fill it in or use Option B / C below.')


In [ ]:
# === OPTION B: Kaggle-attached dataset ======================================
# 1. In the right panel click 'Add Data' and search for a traffic video dataset.
# 2. Update KAGGLE_VIDEO_PATH to the actual mounted path.

# KAGGLE_VIDEO_PATH = '/kaggle/input/<dataset-slug>/video.mp4'
# VIDEO_PATH = KAGGLE_VIDEO_PATH
# assert Path(VIDEO_PATH).exists(), f'Video not found: {VIDEO_PATH}'
# print('✅ Kaggle dataset video:', VIDEO_PATH)

print('Option B: uncomment the lines above after attaching a dataset.')


In [ ]:
# === OPTION C: Synthetic video (no GPU inference needed) ====================
# Generates a synthetic 10-second traffic scene and runs the full pipeline
# with mock detections — no YOLO, no download, no GPU required.
# This is useful to validate pipeline wiring before switching to a real video.

# result = subprocess.run(
#     [sys.executable, 'scripts/dry_run.py', '--frames', '150', '--tracks', '6'],
#     capture_output=False, text=True
# )
# if result.returncode == 0:
#     print('✅ Dry-run complete. Results in outputs/dry_run/')
#     print('   (Skip Steps 4-5; go to Step 6 to inspect outputs.)')

print('Option C: uncomment the block above to run the synthetic dry-run.')


In [ ]:
# === Confirm VIDEO_PATH before proceeding ===================================
# After setting VIDEO_PATH in one of the options above, run this cell.

# Uncomment one of these to set VIDEO_PATH if not already set:
# VIDEO_PATH = 'data/samples/sample_traffic.mp4'
# VIDEO_PATH = '/kaggle/input/<dataset>/video.mp4'

vp = Path(VIDEO_PATH)
if not vp.exists():
    print(f'❌ Video not found: {VIDEO_PATH}')
    print('   Set VIDEO_PATH to a valid .mp4 file using one of the options above.')
else:
    sz = vp.stat().st_size / 1024**2
    print(f'✅ Video confirmed: {VIDEO_PATH} ({sz:.1f} MB)')


## Step 4 — Download YOLOv8 Model Weights

In [ ]:
from ultralytics import YOLO
import shutil, glob

# Options (lightest to heaviest):
#   yolov8n.pt  — nano  (fastest, ~6MB, good for T4)
#   yolov8s.pt  — small (~22MB, recommended balance)
#   yolov8m.pt  — medium (~52MB)
MODEL_NAME = 'yolov8n.pt'

weights_dir = Path('models/checkpoints')
weights_dir.mkdir(parents=True, exist_ok=True)
MODEL_PATH = str(weights_dir / MODEL_NAME)

if not Path(MODEL_PATH).exists():
    print('Downloading', MODEL_NAME, '...')
    YOLO(MODEL_NAME)   # Ultralytics auto-downloads
    matches = glob.glob(
        str(Path.home() / '.cache' / 'ultralytics' / '**' / MODEL_NAME), recursive=True
    )
    if matches:
        shutil.copy(matches[0], MODEL_PATH)

print('✅ Model ready:', MODEL_PATH)


## Step 5 — Run the Full B4DCrash Pipeline

Stages executed:
1. **Perception** — GPU YOLOv8 detection + ByteTrack multi-object tracking
2. **Trajectory reconstruction** — Savitzky-Golay smoothing, velocity/acceleration
3. **Safety metrics** — Multi-timestamp TTC, crossing PET, evasive action detection
4. **Event generation** — Temporal clustering via `EventManager`
5. **Evidence clips** — MP4 clips around each peak conflict timestamp
6. **Hotspot analysis** — Spatial zone risk ranking


In [ ]:
import time

OUTPUT_DIR = '/kaggle/working/outputs'
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print('=' * 60)
print('  B4DCrash Pipeline — Starting')
print('=' * 60)
print('  Video      :', VIDEO_PATH)
print('  Model      :', MODEL_PATH)
print('  Output dir :', OUTPUT_DIR)
print()

t0 = time.time()

ret = subprocess.run(
    [
        sys.executable, 'scripts/nodus_kaggle_runner.py',
        '--video',      VIDEO_PATH,
        '--model',      MODEL_PATH,
        '--output-dir', OUTPUT_DIR,
    ],
    capture_output=False,
)

elapsed = time.time() - t0
if ret.returncode == 0:
    print(f'\n✅ Pipeline complete in {elapsed:.1f}s')
else:
    print(f'\n❌ Pipeline exited with code {ret.returncode} after {elapsed:.1f}s')


## Step 6 — Inspect Output Artifacts

In [ ]:
output_root = Path(OUTPUT_DIR)
files = [f for f in sorted(output_root.rglob('*')) if f.is_file()]

print(f'  {"File":<55} {"Size":>9}')
print(f'  {"-"*55} {"-"*9}')
total = 0
for f in files:
    size_kb = f.stat().st_size / 1024
    total  += size_kb
    rel     = str(f.relative_to(output_root))
    print(f'  {rel:<55} {size_kb:8.1f} KB')
print(f'\n  Total: {total/1024:.2f} MB across {len(files)} files')


## Step 7 — Review Conflict Events

In [ ]:
import pandas as pd

events_csv = Path(OUTPUT_DIR) / 'events' / 'conflict_events.csv'

if not events_csv.exists():
    print('Not found:', events_csv)
else:
    df = pd.read_csv(events_csv)
    print(f'Total conflict events detected: {len(df)}\n')

    if 'severity' in df.columns:
        print('Severity breakdown:')
        print(df['severity'].value_counts().to_string(), '\n')

    if 'risk_score' in df.columns:
        print(f'Risk score — mean: {df["risk_score"].mean():.3f}  max: {df["risk_score"].max():.3f}\n')

    display_cols = [c for c in ['event_id','conflict_type','severity','risk_score','min_ttc','timestamp'] if c in df.columns]
    sort_col     = 'risk_score' if 'risk_score' in df.columns else df.columns[0]
    print('Top events by risk score:')
    print(df.sort_values(sort_col, ascending=False)[display_cols].head(10).to_string(index=False))


## Step 8 — Hotspot Summary

In [ ]:
import pandas as pd

hotspot_csv = Path(OUTPUT_DIR) / 'reports' / 'hotspot_summary.csv'

if not hotspot_csv.exists():
    print('Not found:', hotspot_csv)
else:
    df_h = pd.read_csv(hotspot_csv)
    print(f'Spatial zones analysed: {len(df_h)}\n')
    print(df_h.to_string(index=False))


## Step 9 — Evidence Clips (inline playback)

> Works when running inside the Kaggle notebook environment.


In [ ]:
from IPython.display import Video, display

clips_dir = Path(OUTPUT_DIR) / 'clips'
clips     = sorted(clips_dir.glob('*.mp4')) if clips_dir.exists() else []

if not clips:
    print('No clips found. Clips are generated when conflicts exceed minimum duration.')
else:
    print(f'{len(clips)} evidence clip(s):\n')
    for clip in clips[:3]:
        print(f'  {clip.name}  ({clip.stat().st_size/1024:.1f} KB)')
        display(Video(str(clip), width=640, embed=True))
    if len(clips) > 3:
        print(f'  ... and {len(clips)-3} more in {clips_dir}')


## Step 10 — Agent37 Safety Intelligence Layer

In [ ]:
import json

events_csv_p   = str(Path(OUTPUT_DIR) / 'events' / 'conflict_events.csv')
hotspots_csv_p = str(Path(OUTPUT_DIR) / 'reports' / 'hotspot_summary.csv')

print('Running Agent37 safety intelligence analysis...\n')
agent_result = subprocess.run(
    [
        sys.executable, 'scripts/demo_agent37.py',
        '--events-csv',   events_csv_p,
        '--hotspots-csv', hotspots_csv_p,
    ],
    capture_output=True, text=True,
)
print(agent_result.stdout)
if agent_result.returncode != 0:
    print('STDERR:', agent_result.stderr)

# Load and display structured output
agent_json = Path('outputs/reports/agent37_safety_intelligence.json')
if not agent_json.exists():
    agent_json = Path(OUTPUT_DIR) / 'reports' / 'agent37_safety_intelligence.json'

if agent_json.exists():
    intel = json.loads(agent_json.read_text())
    ra    = intel.get('risk_assessment', {})
    print('=' * 55)
    print('  AGENT37 SAFETY INTELLIGENCE REPORT')
    print('=' * 55)
    print('Risk Level :', ra.get('risk_level', 'N/A'))
    print('Risk Score :', ra.get('overall_risk_score', 'N/A'))
    print('Confidence :', ra.get('confidence', 'N/A'))
    summary = intel.get('executive_summary', '')
    if summary:
        print('\nSummary:\n' + summary)
    for p in intel.get('identified_patterns', []):
        print(f'  [{p.get("severity","?").upper()}] {p.get("description","")}')
    for r in intel.get('recommendations', []):
        print(f'  {r.get("priority","?")}. [{r.get("category","?")}] {r.get("action","")}')
else:
    print('Agent37 output not found. Run Step 5 first.')


## Step 11 — Results Visualisation

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd, numpy as np

events_csv_v  = Path(OUTPUT_DIR) / 'events' / 'conflict_events.csv'
hotspot_csv_v = Path(OUTPUT_DIR) / 'reports' / 'hotspot_summary.csv'

if not events_csv_v.exists():
    print('No conflict_events.csv found. Run the pipeline in Step 5 first.')
else:
    df = pd.read_csv(events_csv_v)

    fig = plt.figure(figsize=(16, 10), facecolor='#0d1117')
    fig.suptitle('B4DCrash — Pipeline Results', color='white', fontsize=18, fontweight='bold')
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

    def sax(ax, title):
        ax.set_facecolor('#161b22')
        ax.tick_params(colors='#adbac7')
        ax.set_title(title, color='white', fontweight='bold', pad=10)
        for sp in ax.spines.values():
            sp.set_edgecolor('#30363d')

    # 1. Severity distribution
    ax1 = fig.add_subplot(gs[0, 0])
    sax(ax1, 'Severity')
    if 'severity' in df.columns:
        pal = {'CRITICAL': '#ff4444', 'HIGH': '#ff8c00', 'MEDIUM': '#ffd700', 'LOW': '#44ff88'}
        c   = df['severity'].value_counts()
        c.plot(kind='bar', ax=ax1, color=[pal.get(s, '#adbac7') for s in c.index], edgecolor='none')
        ax1.set_xlabel('Severity', color='#adbac7')
        ax1.set_ylabel('Count', color='#adbac7')
        ax1.tick_params(axis='x', rotation=0)

    # 2. Conflict type pie
    ax2 = fig.add_subplot(gs[0, 1])
    sax(ax2, 'Conflict Types')
    if 'conflict_type' in df.columns:
        ct = df['conflict_type'].value_counts()
        ct.plot(kind='pie', ax=ax2,
                colors=['#4e9af1', '#f97316', '#a855f7', '#22d3ee'][:len(ct)],
                autopct='%1.0f%%', textprops={'color': 'white'}, startangle=90)
        ax2.set_ylabel('')

    # 3. Risk score histogram
    ax3 = fig.add_subplot(gs[0, 2])
    sax(ax3, 'Risk Score Distribution')
    if 'risk_score' in df.columns:
        ax3.hist(df['risk_score'], bins=20, color='#4e9af1', edgecolor='#30363d', alpha=0.85)
        mean_rs = df['risk_score'].mean()
        ax3.axvline(mean_rs, color='#ff4444', lw=2, linestyle='--', label=f'Mean: {mean_rs:.2f}')
        ax3.legend(facecolor='#161b22', labelcolor='white', edgecolor='#30363d')
        ax3.set_xlabel('Risk Score', color='#adbac7')
        ax3.set_ylabel('Frequency', color='#adbac7')

    # 4. TTC histogram
    ax4 = fig.add_subplot(gs[1, 0])
    sax(ax4, 'Min TTC (seconds)')
    if 'min_ttc' in df.columns:
        ax4.hist(df['min_ttc'].dropna(), bins=15, color='#22d3ee', edgecolor='#30363d', alpha=0.85)
        ax4.axvline(1.5, color='#ff4444', lw=2, linestyle='--', label='Critical (1.5s)')
        ax4.legend(facecolor='#161b22', labelcolor='white', edgecolor='#30363d')
        ax4.set_xlabel('TTC (seconds)', color='#adbac7')
        ax4.set_ylabel('Frequency', color='#adbac7')

    # 5. Hotspot bar chart
    ax5 = fig.add_subplot(gs[1, 1:3])
    sax(ax5, 'Spatial Hotspot Risk Map')
    if hotspot_csv_v.exists():
        df_h  = pd.read_csv(hotspot_csv_v)
        zcol  = next((c for c in df_h.columns if 'zone' in c.lower()), None)
        scol  = next((c for c in df_h.columns if 'risk' in c.lower() or 'score' in c.lower()), None)
        if zcol and scol:
            clrs = plt.cm.YlOrRd(np.linspace(0.3, 0.9, len(df_h)))
            ax5.barh(df_h[zcol].astype(str), df_h[scol], color=clrs, edgecolor='none')
            ax5.set_xlabel('Risk Score', color='#adbac7')
            ax5.tick_params(axis='y', labelcolor='#adbac7')
        else:
            ax5.text(0.5, 0.5, 'Hotspot columns not recognised',
                     ha='center', va='center', color='#adbac7', transform=ax5.transAxes)
    else:
        ax5.text(0.5, 0.5, 'No hotspot data', ha='center', va='center',
                 color='#adbac7', transform=ax5.transAxes)

    out_png = '/kaggle/working/b4dcrash_results.png'
    plt.savefig(out_png, dpi=150, bbox_inches='tight', facecolor='#0d1117')
    plt.show()
    print('✅ Chart saved to', out_png)


## Step 12 — (Optional) Parameter Sensitivity Sweep

Runs Conservative / Moderate / Sensitive configs to compare detection thresholds.


In [ ]:
# Uncomment to run:

# ret = subprocess.run(
#     [sys.executable, 'scripts/nodus_kaggle_runner.py',
#      '--video', VIDEO_PATH,
#      '--model', MODEL_PATH,
#      '--sweep'],
#     capture_output=False,
# )

print('Sweep commented out. Uncomment the block above to run.')


---

## Output Summary

| Artifact | Location | Description |
|----------|----------|-------------|
| `conflict_events.csv` | `outputs/events/` | Near-miss events with TTC, PET, severity, risk score |
| `event_*.mp4` | `outputs/clips/` | Evidence video clips around each conflict |
| `annotated_video.mp4` | `outputs/` | Full video with bounding boxes, tracking, conflict alerts |
| `hotspot_summary.csv` | `outputs/reports/` | Spatial zone risk rankings |
| `agent37_safety_intelligence.json` | `outputs/reports/` | Structured AI safety report |
| `b4dcrash_results.png` | `/kaggle/working/` | Results visualisation chart |

All files are accessible in the **Output** tab on the right side of Kaggle.
